<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/notebook_ipynb/shopee_apify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

# just sample test
# Apify dataset API URL
api_url = 'https://api.apify.com/v2/datasets/tMDnV24byPLyHJSvb/items?token=apify_api_rq6u9yZypsewjMqaBzM07OvcujBo9m2eNoOt'

# Fetch data
response = requests.get(api_url)
if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data)
    print(f'Successfully loaded {len(df)} items.')
    display(df.head())
else:
    print(f'Failed to fetch data: {response.status_code}')
    print(response.text)

Successfully loaded 9 items.


,resultType,batchId,url,status,data,completedAt
0,result,batch_1,https://shopee.co.th/product/1752450934/509594...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-30T02:23:50.203Z
1,result,batch_1,https://shopee.co.th/product/121969451/1918931...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-30T02:23:50.204Z
2,result,batch_1,https://shopee.co.th/product/121969451/2033510...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-30T02:23:50.205Z
3,result,batch_1,https://shopee.co.th/PAO-SUPER-%E0%B8%9C%E0%B8...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-30T02:23:50.206Z
4,result,batch_1,https://shopee.co.th/-%E0%B9%81%E0%B8%9E%E0%B9...,completed,"{'bff_meta': None, 'error': None, 'error_msg':...",2026-07-30T02:23:50.221Z


In [2]:
import datetime
today = datetime.date.today()
print(today)

2026-07-30


In [3]:

# Parse the nested 'data' column
data_parsed = pd.json_normalize(df['data'])

# Map the actual nested column names to your requested labels
columns_to_extract = {
    'data.item.item_id': 'item_id',
    'data.item.shop_id': 'shop_id',
    'data.item.title': 'title',
    'data.item.price': 'price',
    'data.item.price_before_discount': 'price_before_discount'
}

# Filter and rename columns safely
available_cols = [c for c in columns_to_extract.keys() if c in data_parsed.columns]
final_df = data_parsed[available_cols].rename(columns=columns_to_extract)

# Convert prices (divide raw integer price by 100,000)
if 'price' in final_df.columns:
    final_df['price'] = final_df['price'] / 100_000.0
if 'price_before_discount' in final_df.columns:
    final_df['price_before_discount'] = final_df['price_before_discount'] / 100_000.0

# Add Date column
final_df['Date'] = datetime.date.today()

# Save to Excel
output_excel_path = 'shopee_products.xlsx'
final_df.to_excel(output_excel_path, index=False)

print(f"Extracted {len(final_df)} items saved to '{output_excel_path}'")
display(final_df)


Extracted 9 items saved to 'shopee_products.xlsx'


,item_id,shop_id,title,price,price_before_discount,Date
0,50959436125,1752450934,ไฮยีน น้ำยาซักผ้ามิลค์กี้ทัช Hygiene Wash Milk...,65.0,65.0,2026-07-30
1,19189312571,121969451,ไฟน์ไลน์ซักผ้า กลิ่นซันนี่โกลด์ สูตรลดกลิ่นอับ...,122.0,179.0,2026-07-30
2,20335106536,121969451,Fineline ไฟน์ไลน์ ผลิตภัณฑ์ซักผ้าถนอมผ้า พลัสซ...,47.0,89.0,2026-07-30
3,2416923269,30318389,PAO SUPER ผงซักฟอก เปา ซุปเปอร์ 2400 กรัม,139.0,175.0,2026-07-30
4,54104282573,45222406,[แพ็คคู่ 1+1] ไฮยีน เอ็กซ์เพิร์ท แคร์ น้ำยาปรั...,216.0,219.0,2026-07-30
5,899678432,45222406,ไฮยีน เอ็กซ์เพิร์ท แคร์ น้ำยาปรับผ้านุ่มสูตรเข...,55.0,60.0,2026-07-30
6,26461893921,45222406,[แพ็ค 2+1] ไฮยีน เอ็กซ์เพิร์ท แคร์ น้ำยาปรับผ้...,134.0,134.0,2026-07-30
7,1927536731,30318389,PAO ผลิตภัณฑ์ซักผ้า ชนิดน้ำ สูตรเข้มข้น เปา วิ...,89.0,120.0,2026-07-30
8,13356274383,45222406,ไฮยีน เอ็กซ์เพิร์ท แคร์ น้ำยาปรับผ้านุ่มสูตรเข...,125.0,129.0,2026-07-30


In [11]:
import json

target_item_id = 26461893921
target_shop_id = 45222406

# Find the index in final_df that matches our target
match = final_df[(final_df['item_id'] == target_item_id) & (final_df['shop_id'] == target_shop_id)]

if not match.empty:
    idx = match.index[0]
    # Extract the raw content from the 'data' column of the original dataframe
    raw_data_content = df.iloc[idx]['data']

    print(f"--- Raw JSON inside df['data'] for index {idx} ---")
    print(json.dumps(raw_data_content, indent=2, ensure_ascii=False))
else:
    print(f"Could not find item_id {target_item_id} and shop_id {target_shop_id} in final_df.")

--- Raw JSON inside df['data'] for index 6 ---
{
  "bff_meta": null,
  "error": null,
  "error_msg": null,
  "data": {
    "item": {
      "item_id": 26461893921,
      "shop_id": 45222406,
      "item_status": "normal",
      "status": 8,
      "item_type": 0,
      "reference_item_id": "",
      "title": "[แพ็ค 2+1] ไฮยีน เอ็กซ์เพิร์ท แคร์ น้ำยาปรับผ้านุ่มสูตรเข้มข้นพิเศษ 470-480 มล. เลือก 8 กลิ่น",
      "image": "th-11134207-7rasj-m44cngr47n2i4b",
      "label_ids": [
        1014045,
        1383578,
        700640027,
        1400095024,
        1400175096,
        844931064601283,
        844931086908638,
        1908622,
        2073655,
        2058593,
        298643324,
        2103586,
        298938367,
        2158743,
        298983407,
        2213645,
        2213586,
        299103340,
        299073362,
        998091087,
        2098592,
        2108635,
        998301024,
        1400840019,
        1001365,
        1000028,
        1000110,
        700960097,
    

In [13]:
import pandas as pd

# ADDED '.get('data', {})' to correctly navigate the nested JSON
models_data = raw_data_content.get('data', {}).get('item', {}).get('models', [])

# Create a list to hold the extracted option details
options_list = []

for model in models_data:
    # Extract the relevant fields for each selling option
    option_details = {
        'item_id': model.get('item_id'),
        'model_id': model.get('model_id'),
        'name': model.get('name'),
        'name_tr': model.get('name_tr'), # English translation of the option
        'price': model.get('price', 0) / 100_000.0, # Convert raw price format
        'status': model.get('status'),
        'has_stock': model.get('has_stock')
    }
    options_list.append(option_details)

# Convert to a DataFrame for clean visualization
options_df = pd.DataFrame(options_list)

print(f"Found {len(options_df)} selling options for item {target_item_id}:")
display(options_df)

Found 8 selling options for item 26461893921:


,item_id,model_id,name,name_tr,price,status,has_stock
0,26461893921,108460869348,มิลค์กี้ ทัช,Milky Touch,134.0,1,False
1,26461893921,108460869349,ซันคิส บลูมมิ่ง,Sunkiss Blooming,141.0,1,False
2,26461893921,390956347173,เลิฟลี่ บลูม,Lovely Bloom,141.0,1,False
3,26461893921,390956347172,บลูมมิ่ง ทัช,Blooming Touch,134.0,1,False
4,26461893921,390956347174,เลิฟ ทัช,Love Touch,134.0,1,False
5,26461893921,108460869350,ซันไรส์ คิส,Sunrise Kiss,141.0,1,False
6,26461893921,223385571244,พีโอนี บลูม,Peony Bloom,141.0,1,False
7,26461893921,227185098322,แฮปปี้ ซันชายน์,Happy Sunshine,134.0,1,True
